<h1 align="center"> Aplicações em Processamento de Linguagem Natural </h1>
<h3 align="center"> Prof. Fernando Vieira da Silva MSc.</h3>

<h2 align="center"> Trabalho 02</h2>
<h2 align="center">Conversa com personagens do Star Wars</h2>
<h3 align="center"> José Diniz - RA: 183134</h3>

<h3 align="center"> Roberto Rodrigues - RA: 060235</h3>

Crie um chatbot que, dado um texto que você digitou, gere uma resposta no estilo "Star Wars". A resposta deve conter 100 palavras.

Para criar o modelo de geração de texto, você pode utilizar o dataset https://www.kaggle.com/xvivancos/star-wars-movie-scripts

In [1]:
import keras
from keras.models import Sequential
from keras.layers.recurrent import LSTM
from keras.preprocessing.sequence import pad_sequences
from keras.layers.embeddings import Embedding
from keras.layers.core import Activation, Dense, Dropout
from keras.layers.wrappers import TimeDistributed
from keras.layers.core import Dense, Activation
import keras.utils as kutils

Using TensorFlow backend.


Lendo os arquivos que contém os diálogos

In [2]:
all_dialogs = []

for file in ['../input/SW_EpisodeIV.txt','../input/SW_EpisodeV.txt','../input/SW_EpisodeVI.txt']:
    i = 0;
    with open(file) as fl:
        for dlg in fl:
            #"character" "dialogue"
            if(i == 0):
                i+=1
                continue
            #"1" "THREEPIO" "Did you hear that?  They've shut down the main reactor.  We'll be destroyed for sure.  This is madness!"
            #print(dlg.split('" "')[2].replace('"\n',''))
            all_dialogs.append(dlg.split('" "')[2].replace('"\n',''))
            i+=1;

In [3]:
all_dialogs[:10]

["Did you hear that?  They've shut down the main reactor.  We'll be destroyed for sure.  This is madness!",
 "We're doomed!",
 "There'll be no escape for the Princess this time.",
 "What's that?",
 'I should have known better than to trust the logic of a half-sized thermocapsulary dehousing assister...',
 'Hurry up!  Come with me!  What are you waiting for?!  Get in gear!',
 'Artoo! Artoo-Detoo, where are you?',
 'At last!  Where have you been?',
 "They're heading in this direction. What are we going to do?  We'll be sent to the spice mines of Kessel or smashed into who knows what!",
 'Wait a minute, where are you going?']

In [4]:
len(all_dialogs)

2523

Realizando a tokenização e remoção de pontuação.

In [5]:
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
import string

all_sents = [[w.lower() for w in word_tokenize(sen) if not w in string.punctuation] \
             for sen in all_dialogs]

x = []
y = []

print(all_sents[:10])

for sen in all_sents:
    for i in range(1, len(sen)):
        x.append(sen[:i])#Joao
        y.append(sen[i])#Joao da
        
print(x[:10])
print(y[:10])

[['did', 'you', 'hear', 'that', 'they', "'ve", 'shut', 'down', 'the', 'main', 'reactor', 'we', "'ll", 'be', 'destroyed', 'for', 'sure', 'this', 'is', 'madness'], ['we', "'re", 'doomed'], ['there', "'ll", 'be', 'no', 'escape', 'for', 'the', 'princess', 'this', 'time'], ['what', "'s", 'that'], ['i', 'should', 'have', 'known', 'better', 'than', 'to', 'trust', 'the', 'logic', 'of', 'a', 'half-sized', 'thermocapsulary', 'dehousing', 'assister', '...'], ['hurry', 'up', 'come', 'with', 'me', 'what', 'are', 'you', 'waiting', 'for', 'get', 'in', 'gear'], ['artoo', 'artoo-detoo', 'where', 'are', 'you'], ['at', 'last', 'where', 'have', 'you', 'been'], ['they', "'re", 'heading', 'in', 'this', 'direction', 'what', 'are', 'we', 'going', 'to', 'do', 'we', "'ll", 'be', 'sent', 'to', 'the', 'spice', 'mines', 'of', 'kessel', 'or', 'smashed', 'into', 'who', 'knows', 'what'], ['wait', 'a', 'minute', 'where', 'are', 'you', 'going']]
[['did'], ['did', 'you'], ['did', 'you', 'hear'], ['did', 'you', 'hear', '

In [6]:
from sklearn.model_selection import train_test_split
import numpy as np

all_text = [c for sen in x for c in sen]
all_text += [c for c in y]

all_text.append('UNK') # Palavra desconhecida

words = list(set(all_text))
        
word_indexes = {word: index for index, word in enumerate(words)}      

max_features = len(word_indexes)

x = [[word_indexes[c] for c in sen] for sen in x]
y = [word_indexes[c] for c in y]

print(x[:10])
print(y[:10])

y = kutils.to_categorical(y, num_classes=max_features)

maxlen = max([len(sen) for sen in x])

print(maxlen)

[[2546], [2546, 320], [2546, 320, 2413], [2546, 320, 2413, 209], [2546, 320, 2413, 209, 1515], [2546, 320, 2413, 209, 1515, 1427], [2546, 320, 2413, 209, 1515, 1427, 1867], [2546, 320, 2413, 209, 1515, 1427, 1867, 1475], [2546, 320, 2413, 209, 1515, 1427, 1867, 1475, 2051], [2546, 320, 2413, 209, 1515, 1427, 1867, 1475, 2051, 2152]]
[320, 2413, 209, 1515, 1427, 1867, 1475, 2051, 2152, 633]
138


In [7]:
x = pad_sequences(x, maxlen=maxlen)
x = pad_sequences(x, maxlen=maxlen)

print(x[:10,-10:])
print(y[:10,-10:])

[[   0    0    0    0    0    0    0    0    0 2546]
 [   0    0    0    0    0    0    0    0 2546  320]
 [   0    0    0    0    0    0    0 2546  320 2413]
 [   0    0    0    0    0    0 2546  320 2413  209]
 [   0    0    0    0    0 2546  320 2413  209 1515]
 [   0    0    0    0 2546  320 2413  209 1515 1427]
 [   0    0    0 2546  320 2413  209 1515 1427 1867]
 [   0    0 2546  320 2413  209 1515 1427 1867 1475]
 [   0 2546  320 2413  209 1515 1427 1867 1475 2051]
 [2546  320 2413  209 1515 1427 1867 1475 2051 2152]]
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [8]:
embedding_size = 10

model = Sequential()
    
# Add Input Embedding Layer
model.add(Embedding(max_features, embedding_size, input_length=maxlen))
    
# Add Hidden Layer 1 - LSTM Layer
model.add(LSTM(100))
model.add(Dropout(0.1))
    
# Add Output Layer
model.add(Dense(max_features, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam')

Instructions for updating:
Colocations handled automatically by placer.
Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


In [9]:
model.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_1 (Embedding)      (None, 138, 10)           28030     
_________________________________________________________________
lstm_1 (LSTM)                (None, 100)               44400     
_________________________________________________________________
dropout_1 (Dropout)          (None, 100)               0         
_________________________________________________________________
dense_1 (Dense)              (None, 2803)              283103    
Total params: 355,533
Trainable params: 355,533
Non-trainable params: 0
_________________________________________________________________


In [10]:
model.fit(x, y, epochs=20, verbose=5)

Instructions for updating:
Use tf.cast instead.
Epoch 1/20
Epoch 2/20
Epoch 3/20
Epoch 4/20
Epoch 5/20
Epoch 6/20
Epoch 7/20
Epoch 8/20
Epoch 9/20
Epoch 10/20
Epoch 11/20
Epoch 12/20
Epoch 13/20
Epoch 14/20
Epoch 15/20
Epoch 16/20
Epoch 17/20
Epoch 18/20
Epoch 19/20
Epoch 20/20


In [11]:
import pickle

print("Saving model...")
model.save('shak-nlg.h5')

with open('shak-nlg-dict.pkl', 'wb') as handle:
    pickle.dump(word_indexes, handle)

with open('shak-nlg-maxlen.pkl', 'wb') as handle:
    pickle.dump(maxlen, handle)
print("Model Saved!")

Saving model...
Model Saved!


In [12]:
import pickle

model = keras.models.load_model('shak-nlg.h5')
maxlen = pickle.load(open('shak-nlg-maxlen.pkl', 'rb'))
word_indexes = pickle.load(open('shak-nlg-dict.pkl', 'rb'))

Função responsável pela geração do texto randômico

In [13]:
def randomWord(sample_seed):
    sample_seed_vect = np.array([[word_indexes[c] if c in word_indexes.keys() else word_indexes['UNK'] \
                        for c in word_tokenize(sample_seed)]])

    sample_seed_vect = pad_sequences(sample_seed_vect, maxlen=maxlen)

    predicted = model.predict_classes(sample_seed_vect, verbose=0)#para ficar no mesmo tamanh

    def get_word_by_index(index, word_indexes):
        for w, i in word_indexes.items():
            if index == i:
                return w

        return None

    for p in predicted:    
        return get_word_by_index(p, word_indexes)

<h2>Via Input</h2>

In [14]:
text = input()#Oh. Well, yes.
for i in range(1,101,1):
    text+= " " + randomWord(text) + " "
print(text)

StdinNotImplementedError: raw_input was called, but this frontend does not support input requests.

<h2>Via hardcode</h2>

In [15]:
text = "No. I am your father."
for i in range(1,101,1):
    text+= " " + randomWord(text) + " "
print(text)

No. I am your father. you  're  a  small  pilot  for  the  ease  of  the  force  you  have  to  the  emperor  is  strong  the  force  you  be  a  small  defense  directly  to  the  galaxy  it  's  a  long  defense  exhaust  hit  it  's  directly  to  the  galaxy  to  be  swept  on  the  empire  is  spreading  of  the  force  you  were  a  small  defense  chamber  for  the  force  you  be  strong  in  the  force  i  'm  chosen  to  the  academy  generator  the  empire  is  strong  on  the  empire  is  strong  with  you  ...  and  the  emperor  is  strong  in  the  force  i  'm  chosen  to  the 
